# Train Speech Emotion Recognition (MFCC + LSTM)

**Run this notebook in Google Colab.** It downloads the public TESS audio archive, extracts the 40 MFCC features used by the submitted notebook, trains the same LSTM architecture for 50 epochs, evaluates it, and creates a downloadable model bundle.

### What you need to do
1. Open this notebook in Google Colab (`Runtime → Run all`).
2. Accept any prompts to run cells and allow the dataset download to finish.
3. At the end, download `speech_emotion_model_bundle.zip` and extract it into the root of the GitHub project. It contains `models/speech_emotion_lstm.keras`, class labels, and evaluation metrics.

The dataset is TESS (Toronto Emotional Speech Set), 2,800 speech clips with seven labels. It is shared under **CC BY-NC 4.0**, so review the licence before any commercial use. Dataset citation: Pichora-Fuller, M. Kathleen and Kate Dupuis (2020), *Toronto emotional speech set (TESS)*, Borealis, https://doi.org/10.5683/SP2/E8H2MF.

**Reproduction note:** The original notebook averages 40 MFCC coefficients over the time axis before passing them to an LSTM. This training notebook deliberately keeps that preprocessing and the 128-unit LSTM / Dense(64) / Dropout(0.2) / Dense(32) / Dropout(0.2) / Softmax architecture. Scores can differ from the saved presentation due to library versions and random initialisation. The held-out test partition is also used for validation, as in the original notebook, so final portfolio documentation should describe the split accurately.

In [ ]:
# Install compatible dependencies (Google Colab)
%pip -q install "numpy>=1.26,<2.2" "librosa>=0.10.2" "scikit-learn>=1.3" "tensorflow>=2.15,<2.20" pandas matplotlib seaborn requests

In [ ]:
from pathlib import Path
import json, os, shutil, zipfile, time, warnings
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED = 42
tf.keras.utils.set_random_seed(SEED)

WORK_DIR = Path('/content/speech-emotion-recognition')
DATA_DIR = Path('/content/tess_data')
ZIP_PATH = Path('/content/tess.zip')
MODELS_DIR = WORK_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

TESS_URL = 'https://huggingface.co/datasets/myleslinder/tess/resolve/main/data/tess.zip?download=true'
print('TensorFlow:', tf.__version__)
print('Working directory:', WORK_DIR)

In [ ]:
# Download and extract TESS automatically; no Windows-specific path is needed.
audio_files = list(DATA_DIR.rglob('*.wav'))
if len(audio_files) != 2800:
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    print('Downloading the TESS archive (~224 MB)...')
    with requests.get(TESS_URL, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        downloaded = 0
        with open(ZIP_PATH, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f'\rDownloaded {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB', end='')
    print('\nExtracting archive...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        archive.extractall(DATA_DIR)
    audio_files = list(DATA_DIR.rglob('*.wav'))

if len(audio_files) != 2800:
    raise RuntimeError(f'Expected 2,800 WAV files from TESS, but found {len(audio_files)}. Please check the dataset download and rerun this cell.')

# Filename format is speaker_word_emotion.wav, e.g. OAF_back_angry.wav.
records = [{'path': str(p), 'label': p.stem.split('_')[-1].lower()} for p in audio_files]
speech = pd.DataFrame(records).sort_values('path').reset_index(drop=True)
class_names = sorted(speech['label'].unique().tolist())
print('Total audio files:', len(speech))
print('Class labels (model output order):', class_names)
print(speech['label'].value_counts().sort_index())
assert len(class_names) == 7, f'Expected seven emotion classes, got {class_names}'

In [ ]:
# Match the original preprocessing: load up to 3 seconds starting at 0.5 seconds,
# compute 40 MFCCs, average each coefficient over time, then add the final channel axis.
def extract_mfcc(filename):
    y_audio, sample_rate = librosa.load(filename, duration=3, offset=0.5)
    mfcc = np.mean(librosa.feature.mfcc(y=y_audio, sr=sample_rate, n_mfcc=40).T, axis=0)
    return mfcc.astype(np.float32)

features = []
for filename in tqdm(speech['path'], desc='Extracting MFCC features'):
    features.append(extract_mfcc(filename))

X = np.expand_dims(np.asarray(features, dtype=np.float32), axis=-1)
label_to_index = {label: idx for idx, label in enumerate(class_names)}
y_index = speech['label'].map(label_to_index).to_numpy(dtype=np.int64)
y_onehot = tf.keras.utils.to_categorical(y_index, num_classes=len(class_names))

print('X shape:', X.shape)          # expected (2800, 40, 1)
print('y shape:', y_onehot.shape)  # expected (2800, 7)
assert X.shape == (2800, 40, 1)
assert y_onehot.shape == (2800, 7)

In [ ]:
# Reproducible 80/20 stratified split, same general setup as the original notebook.
X_train, X_test, y_train_idx, y_test_idx = train_test_split(
    X, y_index, test_size=0.2, random_state=SEED, stratify=y_index
)
y_train = tf.keras.utils.to_categorical(y_train_idx, num_classes=len(class_names))
y_test = tf.keras.utils.to_categorical(y_test_idx, num_classes=len(class_names))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(40, 1)),
    tf.keras.layers.LSTM(128, return_sequences=False),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=64,
    shuffle=True,
    verbose=1,
)

In [ ]:
# Evaluate and inspect results
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_pred_probs = model.predict(X_test, verbose=0)
y_pred_idx = np.argmax(y_pred_probs, axis=1)

print(f'Training accuracy: {train_acc * 100:.2f}%')
print(f'Test accuracy:     {test_acc * 100:.2f}%')
print(f'Accuracy gap:      {(train_acc - test_acc) * 100:.2f} percentage points')
print('\nClassification report:')
print(classification_report(y_test_idx, y_pred_idx, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['accuracy'], label='Training accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation/test accuracy')
axes[0].set(title='Accuracy by epoch', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='Training loss')
axes[1].plot(history.history['val_loss'], label='Validation/test loss')
axes[1].set(title='Loss by epoch', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test_idx, y_pred_idx, labels=np.arange(len(class_names)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained Keras model plus label/metric metadata.
model_path = MODELS_DIR / 'speech_emotion_lstm.keras'
model.save(model_path)

class_metadata = {
    'class_names': class_names,
    'display_names': {
        'angry': 'Angry',
        'disgust': 'Disgust',
        'fear': 'Fear',
        'happy': 'Happy',
        'neutral': 'Neutral',
        'ps': 'Pleasant Surprise',
        'sad': 'Sad'
    },
    'feature_extraction': 'librosa MFCC n_mfcc=40, mean over time, duration=3 seconds, offset=0.5 seconds',
    'input_shape': [40, 1],
    'dataset': 'TESS (Toronto Emotional Speech Set), 2800 WAV files',
    'seed': SEED
}
(MODELS_DIR / 'class_names.json').write_text(json.dumps(class_metadata, indent=2), encoding='utf-8')

report_dict = classification_report(y_test_idx, y_pred_idx, labels=np.arange(len(class_names)), target_names=class_names, output_dict=True, zero_division=0)
metrics = {
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'train_loss': float(train_loss),
    'test_loss': float(test_loss),
    'accuracy_gap_percentage_points': float((train_acc - test_acc) * 100),
    'test_samples': int(len(y_test_idx)),
    'train_samples': int(len(y_train_idx)),
    'class_names': class_names,
    'classification_report': report_dict,
    'note': 'The test partition is also used as validation during training, matching the original notebook. Treat this score as an in-project result, not a guarantee of real-world performance.'
}
(MODELS_DIR / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

bundle_path = Path('/content/speech_emotion_model_bundle.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for artifact in [model_path, MODELS_DIR / 'class_names.json', MODELS_DIR / 'evaluation_metrics.json']:
        bundle.write(artifact, arcname=str(artifact.relative_to(WORK_DIR)))

print('Model saved:', model_path)
print('Bundle ready:', bundle_path)
print('Bundle files: models/speech_emotion_lstm.keras, models/class_names.json, models/evaluation_metrics.json')

In [ ]:
# Download the artifact bundle to your computer from Google Colab.
from google.colab import files
files.download(str(bundle_path))

## After training

1. Extract `speech_emotion_model_bundle.zip` into the root of your GitHub project (the same folder that contains `README.md`).
2. Confirm the file exists at `models/speech_emotion_lstm.keras`.
3. Run the Streamlit app using `streamlit run app/streamlit_app.py`.
4. Do not commit the TESS audio files to GitHub. The dataset licence is CC BY-NC 4.0; include its citation and licence note in your project documentation.

The original labels use `ps` for **pleasant surprise**. This is not a separate “PS” emotion; it is the short dataset label for pleasant surprise.